In [53]:
import pandas as pd 
import numpy as np
import ot
import itertools
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize

# SAM-Med3D

In [81]:
df_feats = pd.read_csv("../../local/output/combined_data_sammed.csv")
# Assume df_feats has 'client' column and 'Feature1'..'Feature384'
features_cols = [f'Feature{i}' for i in range(1, 385)]

# Get unique cohorts
cohorts = df_feats['cohort'].unique()

# Prepare results list
results = []

In [ ]:
# Compute OT distance for each pair
for cohort1, cohort2 in itertools.combinations(cohorts, 2):
    print(f'Computing distance between {cohort1} and {cohort2}')
    X = df_feats[df_feats['cohort'] == cohort1][features_cols].to_numpy()
    Y = df_feats[df_feats['cohort'] == cohort2][features_cols].to_numpy()
    
    # L2-normalize per sample
    X_norm = normalize(X, norm='l2', axis=1)
    Y_norm = normalize(Y, norm='l2', axis=1)
    
    # Uniform weights
    a = np.ones((X_norm.shape[0],)) / X_norm.shape[0]
    b = np.ones((Y_norm.shape[0],)) / Y_norm.shape[0]
    
    # Cost matrix (squared Euclidean distance)
    M = ot.dist(X_norm, Y_norm, metric='euclidean')**2
    
    # Wasserstein-2 distance
    w2 = np.sqrt(ot.emd2(a, b, M))
    
    results.append({'pair': f'{cohort1}_{cohort2}', 'distance': w2})

In [83]:
df_results = pd.DataFrame(results)
distance_matrix = pd.DataFrame(0, index=cohorts, columns=cohorts, dtype=float)

In [ ]:
for row in results:
    c1, c2 = row['pair'].split('_')
    distance_matrix.loc[c1, c2] = row['distance']
    distance_matrix.loc[c2, c1] = row['distance']  # symmetric

mask = np.triu(np.ones_like(distance_matrix, dtype=bool))
plt.figure(figsize=(12, 10))
sns.heatmap(distance_matrix, annot=True, fmt=".2f", cmap='viridis',
            mask=mask, cbar_kws={'label': 'Wasserstein-2 Distance'})
plt.title('Wasserstein-2 Distance Between Cohorts (Lower Triangle Only)')
plt.show()


In [ ]:
# Suppose 'distance_matrix' is already built with labels like 'ClientA_CN', 'ClientB_AD'

# Get unique (cohort, label) groups
groups = df_feats.groupby(['cohort', 'label'])

group_keys = list(groups.groups.keys())  # list of tuples (cohort, label)

results = []

# Compute Wasserstein-2 distance between all pairs of groups
for (c1, l1), (c2, l2) in itertools.combinations(group_keys, 2):
    X = groups.get_group((c1, l1))[features_cols].to_numpy()
    Y = groups.get_group((c2, l2))[features_cols].to_numpy()
    
    # L2-normalize per sample
    X_norm = normalize(X, norm='l2', axis=1)
    Y_norm = normalize(Y, norm='l2', axis=1)
    
    # Uniform weights
    a = np.ones((X_norm.shape[0],)) / X_norm.shape[0]
    b = np.ones((Y_norm.shape[0],)) / Y_norm.shape[0]
    
    # Cost matrix (squared Euclidean)
    M = ot.dist(X_norm, Y_norm, metric='euclidean')**2
    
    # Wasserstein-2 distance
    w2 = np.sqrt(ot.emd2(a, b, M))
    
    results.append({
        'pair': f'{c1}_{l1}-{c2}_{l2}',
        'distance': w2
    })

# Convert to DataFrame and save

# Convert to DataFrame and save
df_results = pd.DataFrame(results)

# Optional: create a heatmap
# Build a square distance matrix
all_keys = [f'{c}_{l}' for c, l in group_keys]
distance_matrix = pd.DataFrame(0, index=all_keys, columns=all_keys, dtype=float)

for row in results:
    g1, g2 = row['pair'].split('-')
    distance_matrix.loc[g1, g2] = row['distance']
    distance_matrix.loc[g2, g1] = row['distance']  # symmetric

plt.figure(figsize=(12, 10))
sns.heatmap(distance_matrix, annot=True, fmt=".2f", cmap='viridis')
plt.title('Wasserstein-2 Distance Between Cohort-Label Groups')
plt.show()

In [ ]:
# Map label numbers to class names
label_map = {0: 'CN', 1: 'AD'}

# Build group keys with class names
all_keys = [f'{c}_{label_map[l]}' for c, l in group_keys]

# Order keys: all CN first, then all AD
all_keys_sorted = sorted(all_keys, key=lambda x: x.split('_')[-1])

# Build a square distance matrix
distance_matrix = pd.DataFrame(0, index=all_keys_sorted, columns=all_keys_sorted, dtype=float)

for row in results:
    g1, g2 = row['pair'].split('-')
    # Replace numeric labels with class names
    g1_name = f'{g1.split("_")[0]}_{label_map[int(g1.split("_")[1])]}'
    g2_name = f'{g2.split("_")[0]}_{label_map[int(g2.split("_")[1])]}'
    
    distance_matrix.loc[g1_name, g2_name] = row['distance']
    distance_matrix.loc[g2_name, g1_name] = row['distance']  # symmetric

# Create a mask for the upper triangle
mask = np.triu(np.ones_like(distance_matrix, dtype=bool))

plt.figure(figsize=(12, 10))
sns.heatmap(distance_matrix, annot=True, fmt=".2f", cmap='viridis',
            mask=mask, cbar_kws={'label': 'Wasserstein-2 Distance'})
plt.title('Wasserstein-2 Distance Between Client-Label Groups (Lower Triangle Only)')
plt.show()

In [ ]:
# Suppose 'distance_matrix' is already built with labels like 'ClientA_CN', 'ClientB_AD'
labels = distance_matrix.index.tolist()
labels = [lbl.replace("_", "-") for lbl in labels]  # Replace underscore with dash for better parsing
# Extract class names from group keys
classes = [lbl.split('-')[-1] for lbl in labels]

from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Polygon, Patch

# Sequential single-hue (gray) colormap: light = small distance, dark = large distance
distance_cmap = LinearSegmentedColormap.from_list(
    'seq_gray', ["#babebe", "#909a99", "#717879", "#4b4f51", "#0b0b0b"]
)

# Categorical colors for the group-pair border only (never used for cell fill)
group_border_colors = {
    'CN_CN': '#1b9e77',  # blue
    'AD_AD': '#7570b3',  # orange
    'CN_AD': '#d95f02',  # aqua (mixed comparison)
}
group_labels = {
    'CN_CN': 'CN vs CN',
    'AD_AD': 'AD vs AD',
    'CN_AD': 'CN vs AD',
}

def group_key(cls_i, cls_j):
    if cls_i == cls_j:
        return f'{cls_i}_{cls_j}'
    return 'CN_AD'

# Inset each border by this many cell-widths so that borders of adjacent
# blocks don't sit exactly on top of each other (which hid whichever was
# drawn first) -- each block's outline shrinks toward its own interior,
# leaving a small gap between neighboring borders.
BORDER_MARGIN = 0.04

def staircase_vertices(r0, c0, k, margin=0.0):
    """Boundary of the strictly-lower-triangular cells of a k x k block
    (block-local origin at row r0, col c0), following the cell edges,
    inset inward by `margin`."""
    m = margin
    verts = [(c0 + m, r0 + 1 + m)]
    for j in range(k - 2):
        verts.append((c0 + j + 1 - m, r0 + j + 1 + m))
        verts.append((c0 + j + 1 - m, r0 + j + 2 + m))
    verts.append((c0 + k - 1 - m, r0 + k - 1 + m))
    verts.append((c0 + k - 1 - m, r0 + k - m))
    verts.append((c0 + m, r0 + k - m))
    return verts

# Mask upper triangle
mask = np.triu(np.ones_like(distance_matrix, dtype=bool))

plt.figure(figsize=(12, 10))
ax = sns.heatmap(distance_matrix, annot=True, fmt=".2f", mask=mask,
                  cmap=distance_cmap, square=True,
                  linewidths=.5, linecolor='white',
                  cbar_kws={'label': '2-Wasserstein Distance'},
                  annot_kws={"size": 16})

# Keep annotation text legible against both light and dark cells
dist_min = distance_matrix.values.min()
dist_max = distance_matrix.values.max()
norm_dist = (distance_matrix.values - dist_min) / (dist_max - dist_min + 1e-8)
for text in ax.texts:
    x, y = text.get_position()
    col, row = int(x), int(y)
    text.set_color('white' if norm_dist[row, col] > 0.5 else 'black')

# Outline each contiguous CN/AD block with a color that identifies the
# comparison type; position already groups them, the border just labels it.
# Diagonal blocks are only half-visible (the mask hides their upper triangle),
# so their outline follows the staircase of visible cells rather than a smooth
# diagonal (which would leave gaps between the line and the cells). clip_on is
# disabled so the outer edges aren't half-clipped by the axes boundary.
boundaries = [0] + [i for i in range(1, len(classes)) if classes[i] != classes[i - 1]] + [len(classes)]
for bi in range(len(boundaries) - 1):
    for bj in range(len(boundaries) - 1):
        if bi < bj:
            continue  # entirely in the masked upper triangle
        r0, r1 = boundaries[bi], boundaries[bi + 1]
        c0, c1 = boundaries[bj], boundaries[bj + 1]
        key = group_key(classes[r0], classes[c0])
        edgecolor = group_border_colors[key]
        if bi == bj:
            k = r1 - r0
            if k < 2:
                continue  # single-cell block: entirely masked, nothing to outline
            verts = staircase_vertices(r0, c0, k, BORDER_MARGIN)
            ax.add_patch(Polygon(verts, closed=True, fill=False, edgecolor=edgecolor,
                                  lw=4, zorder=10, clip_on=False))
        else:
            m = BORDER_MARGIN
            ax.add_patch(plt.Rectangle((c0 + m, r0 + m), (c1 - c0) - 2 * m, (r1 - r0) - 2 * m,
                                        fill=False, edgecolor=edgecolor, lw=4, zorder=10, clip_on=False))

# Legend explaining the border colors
legend_handles = [Patch(facecolor='none', edgecolor=color, linewidth=3, label=group_labels[key])
                  for key, color in group_border_colors.items()]
ax.legend(handles=legend_handles, loc='upper right', frameon=True, framealpha=0.9,
          fontsize=12)

plt.xticks(np.arange(len(labels))+0.5, labels, rotation=90, fontsize=14)
plt.yticks(np.arange(len(labels))+0.5, labels, rotation=0, fontsize=14)
plt.title('SAM-Med3D', fontsize=20)
plt.savefig("../../local/output/SAM-Med3D_Wasserstein2_distance_heatmap.png", bbox_inches='tight', dpi=300)
plt.show()

# BrainIAC

In [102]:
df_feats = pd.read_csv("../../local/output/combined_data_brainiac.csv")
# Assume df_feats has 'client' column and 'Feature1'..'Feature384'
features_cols = [f'Feature{i}' for i in range(1, 769)]

# Get unique cohorts
cohorts = df_feats['cohort'].unique()

# Prepare results list
results = []

In [ ]:
# Compute OT distance for each pair
for cohort1, cohort2 in itertools.combinations(cohorts, 2):
    print(f'Computing distance between {cohort1} and {cohort2}')
    X = df_feats[df_feats['cohort'] == cohort1][features_cols].to_numpy()
    Y = df_feats[df_feats['cohort'] == cohort2][features_cols].to_numpy()
    
    # L2-normalize per sample
    X_norm = normalize(X, norm='l2', axis=1)
    Y_norm = normalize(Y, norm='l2', axis=1)
    
    # Uniform weights
    a = np.ones((X_norm.shape[0],)) / X_norm.shape[0]
    b = np.ones((Y_norm.shape[0],)) / Y_norm.shape[0]
    
    # Cost matrix (squared Euclidean distance)
    M = ot.dist(X_norm, Y_norm, metric='euclidean')**2
    
    # Wasserstein-2 distance
    w2 = np.sqrt(ot.emd2(a, b, M))
    
    results.append({'pair': f'{cohort1}_{cohort2}', 'distance': w2})

In [104]:
df_results = pd.DataFrame(results)
distance_matrix = pd.DataFrame(0, index=cohorts, columns=cohorts, dtype=float)

In [ ]:
for row in results:
    c1, c2 = row['pair'].split('_')
    distance_matrix.loc[c1, c2] = row['distance']
    distance_matrix.loc[c2, c1] = row['distance']  # symmetric

mask = np.triu(np.ones_like(distance_matrix, dtype=bool))
plt.figure(figsize=(12, 10))
sns.heatmap(distance_matrix, annot=True, fmt=".2f", cmap='viridis',
            mask=mask, cbar_kws={'label': 'Wasserstein-2 Distance'})
plt.title('Wasserstein-2 Distance Between Cohorts (Lower Triangle Only)')
plt.show()


In [ ]:
# Suppose 'distance_matrix' is already built with labels like 'ClientA_CN', 'ClientB_AD'

# Get unique (cohort, label) groups
groups = df_feats.groupby(['cohort', 'label'])

group_keys = list(groups.groups.keys())  # list of tuples (cohort, label)

results = []

# Compute Wasserstein-2 distance between all pairs of groups
for (c1, l1), (c2, l2) in itertools.combinations(group_keys, 2):
    X = groups.get_group((c1, l1))[features_cols].to_numpy()
    Y = groups.get_group((c2, l2))[features_cols].to_numpy()
    
    # L2-normalize per sample
    X_norm = normalize(X, norm='l2', axis=1)
    Y_norm = normalize(Y, norm='l2', axis=1)
    
    # Uniform weights
    a = np.ones((X_norm.shape[0],)) / X_norm.shape[0]
    b = np.ones((Y_norm.shape[0],)) / Y_norm.shape[0]
    
    # Cost matrix (squared Euclidean)
    M = ot.dist(X_norm, Y_norm, metric='euclidean')**2
    
    # Wasserstein-2 distance
    w2 = np.sqrt(ot.emd2(a, b, M))
    
    results.append({
        'pair': f'{c1}_{l1}-{c2}_{l2}',
        'distance': w2
    })

# Convert to DataFrame and save

# Convert to DataFrame and save
df_results = pd.DataFrame(results)

# Optional: create a heatmap
# Build a square distance matrix
all_keys = [f'{c}_{l}' for c, l in group_keys]
distance_matrix = pd.DataFrame(0, index=all_keys, columns=all_keys, dtype=float)

for row in results:
    g1, g2 = row['pair'].split('-')
    distance_matrix.loc[g1, g2] = row['distance']
    distance_matrix.loc[g2, g1] = row['distance']  # symmetric

plt.figure(figsize=(12, 10))
sns.heatmap(distance_matrix, annot=True, fmt=".2f", cmap='viridis')
plt.title('Wasserstein-2 Distance Between Cohort-Label Groups')
plt.show()

In [ ]:
# Map label numbers to class names
label_map = {0: 'CN', 1: 'AD'}

# Build group keys with class names
all_keys = [f'{c}_{label_map[l]}' for c, l in group_keys]

# Order keys: all CN first, then all AD
all_keys_sorted = sorted(all_keys, key=lambda x: x.split('_')[-1])

# Build a square distance matrix
distance_matrix = pd.DataFrame(0, index=all_keys_sorted, columns=all_keys_sorted, dtype=float)

for row in results:
    g1, g2 = row['pair'].split('-')
    # Replace numeric labels with class names
    g1_name = f'{g1.split("_")[0]}_{label_map[int(g1.split("_")[1])]}'
    g2_name = f'{g2.split("_")[0]}_{label_map[int(g2.split("_")[1])]}'
    
    distance_matrix.loc[g1_name, g2_name] = row['distance']
    distance_matrix.loc[g2_name, g1_name] = row['distance']  # symmetric

# Create a mask for the upper triangle
mask = np.triu(np.ones_like(distance_matrix, dtype=bool))

plt.figure(figsize=(12, 10))
sns.heatmap(distance_matrix, annot=True, fmt=".2f", cmap='viridis',
            mask=mask, cbar_kws={'label': 'Wasserstein-2 Distance'})
plt.title('Wasserstein-2 Distance Between Client-Label Groups (Lower Triangle Only)')
plt.show()

In [ ]:
# Suppose 'distance_matrix' is already built with labels like 'ClientA_CN', 'ClientB_AD'
labels = distance_matrix.index.tolist()
labels = [lbl.replace("_", "-") for lbl in labels]  # Replace underscore with dash for better parsing
# Extract class names from group keys
classes = [lbl.split('-')[-1] for lbl in labels]

from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Polygon, Patch

# Sequential single-hue (blue) colormap: light = small distance, dark = large distance
# Sequential single-hue (gray) colormap: light = small distance, dark = large distance
distance_cmap = LinearSegmentedColormap.from_list(
    'seq_gray', ["#babebe", "#909a99", "#717879", "#4b4f51", "#0b0b0b"]
)

# Categorical colors for the group-pair border only (never used for cell fill)
group_border_colors = {
    'CN_CN': '#1b9e77',  # blue
    'AD_AD': '#7570b3',  # orange
    'CN_AD': '#d95f02',  # aqua (mixed comparison)
}
group_labels = {
    'CN_CN': 'CN vs CN',
    'AD_AD': 'AD vs AD',
    'CN_AD': 'CN vs AD',
}

def group_key(cls_i, cls_j):
    if cls_i == cls_j:
        return f'{cls_i}_{cls_j}'
    return 'CN_AD'

# Inset each border by this many cell-widths so that borders of adjacent
# blocks don't sit exactly on top of each other (which hid whichever was
# drawn first) -- each block's outline shrinks toward its own interior,
# leaving a small gap between neighboring borders.
BORDER_MARGIN = 0.04

def staircase_vertices(r0, c0, k, margin=0.0):
    """Boundary of the strictly-lower-triangular cells of a k x k block
    (block-local origin at row r0, col c0), following the cell edges,
    inset inward by `margin`."""
    m = margin
    verts = [(c0 + m, r0 + 1 + m)]
    for j in range(k - 2):
        verts.append((c0 + j + 1 - m, r0 + j + 1 + m))
        verts.append((c0 + j + 1 - m, r0 + j + 2 + m))
    verts.append((c0 + k - 1 - m, r0 + k - 1 + m))
    verts.append((c0 + k - 1 - m, r0 + k - m))
    verts.append((c0 + m, r0 + k - m))
    return verts

# Mask upper triangle
mask = np.triu(np.ones_like(distance_matrix, dtype=bool))

plt.figure(figsize=(12, 10))
ax = sns.heatmap(distance_matrix, annot=True, fmt=".2f", mask=mask,
                  cmap=distance_cmap, square=True,
                  linewidths=.5, linecolor='white',
                  cbar_kws={'label': '2-Wasserstein Distance'},
                  annot_kws={"size": 16})

# Keep annotation text legible against both light and dark cells
dist_min = distance_matrix.values.min()
dist_max = distance_matrix.values.max()
norm_dist = (distance_matrix.values - dist_min) / (dist_max - dist_min + 1e-8)
for text in ax.texts:
    x, y = text.get_position()
    col, row = int(x), int(y)
    text.set_color('white' if norm_dist[row, col] > 0.5 else 'black')

# Outline each contiguous CN/AD block with a color that identifies the
# comparison type; position already groups them, the border just labels it.
# Diagonal blocks are only half-visible (the mask hides their upper triangle),
# so their outline follows the staircase of visible cells rather than a smooth
# diagonal (which would leave gaps between the line and the cells). clip_on is
# disabled so the outer edges aren't half-clipped by the axes boundary.
boundaries = [0] + [i for i in range(1, len(classes)) if classes[i] != classes[i - 1]] + [len(classes)]
for bi in range(len(boundaries) - 1):
    for bj in range(len(boundaries) - 1):
        if bi < bj:
            continue  # entirely in the masked upper triangle
        r0, r1 = boundaries[bi], boundaries[bi + 1]
        c0, c1 = boundaries[bj], boundaries[bj + 1]
        key = group_key(classes[r0], classes[c0])
        edgecolor = group_border_colors[key]
        if bi == bj:
            k = r1 - r0
            if k < 2:
                continue  # single-cell block: entirely masked, nothing to outline
            verts = staircase_vertices(r0, c0, k, BORDER_MARGIN)
            ax.add_patch(Polygon(verts, closed=True, fill=False, edgecolor=edgecolor,
                                  lw=4, zorder=10, clip_on=False))
        else:
            m = BORDER_MARGIN
            ax.add_patch(plt.Rectangle((c0 + m, r0 + m), (c1 - c0) - 2 * m, (r1 - r0) - 2 * m,
                                        fill=False, edgecolor=edgecolor, lw=4, zorder=10, clip_on=False))

# Legend explaining the border colors
legend_handles = [Patch(facecolor='none', edgecolor=color, linewidth=3, label=group_labels[key])
                  for key, color in group_border_colors.items()]
ax.legend(handles=legend_handles, loc='upper right', frameon=True, framealpha=0.9,
          fontsize=12)

plt.xticks(np.arange(len(labels))+0.5, labels, rotation=90, fontsize=14)
plt.yticks(np.arange(len(labels))+0.5, labels, rotation=0, fontsize=14)
plt.title('BrainIAC', fontsize=20)
plt.savefig("../../local/output/BrainIAC_Wasserstein2_distance_heatmap.png", bbox_inches='tight', dpi=300)
plt.show()

# 3DINO

In [109]:
df_feats = pd.read_csv("../../local/output/combined_data_dino.csv")
# Assume df_feats has 'client' column and 'Feature1'..'Feature384'
features_cols = [f'Feature{i}' for i in range(1, 1025)]

# Get unique cohorts
cohorts = df_feats['cohort'].unique()

# Prepare results list
results = []

In [ ]:
# Compute OT distance for each pair
for cohort1, cohort2 in itertools.combinations(cohorts, 2):
    print(f'Computing distance between {cohort1} and {cohort2}')
    X = df_feats[df_feats['cohort'] == cohort1][features_cols].to_numpy()
    Y = df_feats[df_feats['cohort'] == cohort2][features_cols].to_numpy()
    
    # L2-normalize per sample
    X_norm = normalize(X, norm='l2', axis=1)
    Y_norm = normalize(Y, norm='l2', axis=1)
    
    # Uniform weights
    a = np.ones((X_norm.shape[0],)) / X_norm.shape[0]
    b = np.ones((Y_norm.shape[0],)) / Y_norm.shape[0]
    
    # Cost matrix (squared Euclidean distance)
    M = ot.dist(X_norm, Y_norm, metric='euclidean')**2
    
    # Wasserstein-2 distance
    w2 = np.sqrt(ot.emd2(a, b, M))
    
    results.append({'pair': f'{cohort1}_{cohort2}', 'distance': w2})

In [111]:
df_results = pd.DataFrame(results)
distance_matrix = pd.DataFrame(0, index=cohorts, columns=cohorts, dtype=float)

In [ ]:
for row in results:
    c1, c2 = row['pair'].split('_')
    distance_matrix.loc[c1, c2] = row['distance']
    distance_matrix.loc[c2, c1] = row['distance']  # symmetric

mask = np.triu(np.ones_like(distance_matrix, dtype=bool))
plt.figure(figsize=(12, 10))
sns.heatmap(distance_matrix, annot=True, fmt=".2f", cmap='viridis',
            mask=mask, cbar_kws={'label': 'Wasserstein-2 Distance'})
plt.title('Wasserstein-2 Distance Between Cohorts (Lower Triangle Only)')
plt.show()


In [ ]:
# Suppose 'distance_matrix' is already built with labels like 'ClientA_CN', 'ClientB_AD'

# Get unique (cohort, label) groups
groups = df_feats.groupby(['cohort', 'label'])

group_keys = list(groups.groups.keys())  # list of tuples (cohort, label)

results = []

# Compute Wasserstein-2 distance between all pairs of groups
for (c1, l1), (c2, l2) in itertools.combinations(group_keys, 2):
    X = groups.get_group((c1, l1))[features_cols].to_numpy()
    Y = groups.get_group((c2, l2))[features_cols].to_numpy()
    
    # L2-normalize per sample
    X_norm = normalize(X, norm='l2', axis=1)
    Y_norm = normalize(Y, norm='l2', axis=1)
    
    # Uniform weights
    a = np.ones((X_norm.shape[0],)) / X_norm.shape[0]
    b = np.ones((Y_norm.shape[0],)) / Y_norm.shape[0]
    
    # Cost matrix (squared Euclidean)
    M = ot.dist(X_norm, Y_norm, metric='euclidean')**2
    
    # Wasserstein-2 distance
    w2 = np.sqrt(ot.emd2(a, b, M))
    
    results.append({
        'pair': f'{c1}_{l1}-{c2}_{l2}',
        'distance': w2
    })

# Convert to DataFrame and save

# Convert to DataFrame and save
df_results = pd.DataFrame(results)

# Optional: create a heatmap
# Build a square distance matrix
all_keys = [f'{c}_{l}' for c, l in group_keys]
distance_matrix = pd.DataFrame(0, index=all_keys, columns=all_keys, dtype=float)

for row in results:
    g1, g2 = row['pair'].split('-')
    distance_matrix.loc[g1, g2] = row['distance']
    distance_matrix.loc[g2, g1] = row['distance']  # symmetric

plt.figure(figsize=(12, 10))
sns.heatmap(distance_matrix, annot=True, fmt=".2f", cmap='viridis')
plt.title('Wasserstein-2 Distance Between Cohort-Label Groups')
plt.show()

In [ ]:
# Map label numbers to class names
label_map = {0: 'CN', 1: 'AD'}

# Build group keys with class names
all_keys = [f'{c}_{label_map[l]}' for c, l in group_keys]

# Order keys: all CN first, then all AD
all_keys_sorted = sorted(all_keys, key=lambda x: x.split('_')[-1])

# Build a square distance matrix
distance_matrix = pd.DataFrame(0, index=all_keys_sorted, columns=all_keys_sorted, dtype=float)

for row in results:
    g1, g2 = row['pair'].split('-')
    # Replace numeric labels with class names
    g1_name = f'{g1.split("_")[0]}_{label_map[int(g1.split("_")[1])]}'
    g2_name = f'{g2.split("_")[0]}_{label_map[int(g2.split("_")[1])]}'
    
    distance_matrix.loc[g1_name, g2_name] = row['distance']
    distance_matrix.loc[g2_name, g1_name] = row['distance']  # symmetric

# Create a mask for the upper triangle
mask = np.triu(np.ones_like(distance_matrix, dtype=bool))

plt.figure(figsize=(12, 10))
sns.heatmap(distance_matrix, annot=True, fmt=".2f", cmap='viridis',
            mask=mask, cbar_kws={'label': 'Wasserstein-2 Distance'})
plt.title('Wasserstein-2 Distance Between Client-Label Groups (Lower Triangle Only)')
plt.show()

In [ ]:
# Suppose 'distance_matrix' is already built with labels like 'ClientA_CN', 'ClientB_AD'
labels = distance_matrix.index.tolist()
labels = [lbl.replace("_", "-") for lbl in labels]  # Replace underscore with dash for better parsing
# Extract class names from group keys
classes = [lbl.split('-')[-1] for lbl in labels]

from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Polygon, Patch

# Sequential single-hue (blue) colormap: light = small distance, dark = large distance
# Sequential single-hue (gray) colormap: light = small distance, dark = large distance
distance_cmap = LinearSegmentedColormap.from_list(
    'seq_gray', ["#babebe", "#909a99", "#717879", "#4b4f51", "#0b0b0b"]
)

# Categorical colors for the group-pair border only (never used for cell fill)
group_border_colors = {
    'CN_CN': '#1b9e77',  # blue
    'AD_AD': '#7570b3',  # orange
    'CN_AD': '#d95f02',  # aqua (mixed comparison)
}
group_labels = {
    'CN_CN': 'CN vs CN',
    'AD_AD': 'AD vs AD',
    'CN_AD': 'CN vs AD',
}

def group_key(cls_i, cls_j):
    if cls_i == cls_j:
        return f'{cls_i}_{cls_j}'
    return 'CN_AD'

# Inset each border by this many cell-widths so that borders of adjacent
# blocks don't sit exactly on top of each other (which hid whichever was
# drawn first) -- each block's outline shrinks toward its own interior,
# leaving a small gap between neighboring borders.
BORDER_MARGIN = 0.04

def staircase_vertices(r0, c0, k, margin=0.0):
    """Boundary of the strictly-lower-triangular cells of a k x k block
    (block-local origin at row r0, col c0), following the cell edges,
    inset inward by `margin`."""
    m = margin
    verts = [(c0 + m, r0 + 1 + m)]
    for j in range(k - 2):
        verts.append((c0 + j + 1 - m, r0 + j + 1 + m))
        verts.append((c0 + j + 1 - m, r0 + j + 2 + m))
    verts.append((c0 + k - 1 - m, r0 + k - 1 + m))
    verts.append((c0 + k - 1 - m, r0 + k - m))
    verts.append((c0 + m, r0 + k - m))
    return verts

# Mask upper triangle
mask = np.triu(np.ones_like(distance_matrix, dtype=bool))

plt.figure(figsize=(12, 10))
ax = sns.heatmap(distance_matrix, annot=True, fmt=".2f", mask=mask,
                  cmap=distance_cmap, square=True,
                  linewidths=.5, linecolor='white',
                  cbar_kws={'label': '2-Wasserstein Distance'},
                  annot_kws={"size": 16})

# Keep annotation text legible against both light and dark cells
dist_min = distance_matrix.values.min()
dist_max = distance_matrix.values.max()
norm_dist = (distance_matrix.values - dist_min) / (dist_max - dist_min + 1e-8)
for text in ax.texts:
    x, y = text.get_position()
    col, row = int(x), int(y)
    text.set_color('white' if norm_dist[row, col] > 0.5 else 'black')

# Outline each contiguous CN/AD block with a color that identifies the
# comparison type; position already groups them, the border just labels it.
# Diagonal blocks are only half-visible (the mask hides their upper triangle),
# so their outline follows the staircase of visible cells rather than a smooth
# diagonal (which would leave gaps between the line and the cells). clip_on is
# disabled so the outer edges aren't half-clipped by the axes boundary.
boundaries = [0] + [i for i in range(1, len(classes)) if classes[i] != classes[i - 1]] + [len(classes)]
for bi in range(len(boundaries) - 1):
    for bj in range(len(boundaries) - 1):
        if bi < bj:
            continue  # entirely in the masked upper triangle
        r0, r1 = boundaries[bi], boundaries[bi + 1]
        c0, c1 = boundaries[bj], boundaries[bj + 1]
        key = group_key(classes[r0], classes[c0])
        edgecolor = group_border_colors[key]
        if bi == bj:
            k = r1 - r0
            if k < 2:
                continue  # single-cell block: entirely masked, nothing to outline
            verts = staircase_vertices(r0, c0, k, BORDER_MARGIN)
            ax.add_patch(Polygon(verts, closed=True, fill=False, edgecolor=edgecolor,
                                  lw=4, zorder=10, clip_on=False))
        else:
            m = BORDER_MARGIN
            ax.add_patch(plt.Rectangle((c0 + m, r0 + m), (c1 - c0) - 2 * m, (r1 - r0) - 2 * m,
                                        fill=False, edgecolor=edgecolor, lw=4, zorder=10, clip_on=False))

# Legend explaining the border colors
legend_handles = [Patch(facecolor='none', edgecolor=color, linewidth=3, label=group_labels[key])
                  for key, color in group_border_colors.items()]
ax.legend(handles=legend_handles, loc='upper right', frameon=True, framealpha=0.9,
          fontsize=12)

plt.xticks(np.arange(len(labels))+0.5, labels, rotation=90, fontsize=14)
plt.yticks(np.arange(len(labels))+0.5, labels, rotation=0, fontsize=14)
plt.title('3DINO', fontsize=20)
plt.savefig("../../local/output/3DINO_Wasserstein2_distance_heatmap.png", bbox_inches='tight', dpi=300)
plt.show()

# Med-SAM2

In [116]:
df_feats = pd.read_csv("../../local/output/combined_data_medsam2.csv")
# Assume df_feats has 'client' column and 'Feature1'..'Feature384'
features_cols = [f'Feature{i}' for i in range(1, 257)]

# Get unique cohorts
cohorts = df_feats['cohort'].unique()

# Prepare results list
results = []

In [ ]:
# Compute OT distance for each pair
for cohort1, cohort2 in itertools.combinations(cohorts, 2):
    print(f'Computing distance between {cohort1} and {cohort2}')
    X = df_feats[df_feats['cohort'] == cohort1][features_cols].to_numpy()
    Y = df_feats[df_feats['cohort'] == cohort2][features_cols].to_numpy()
    
    # L2-normalize per sample
    X_norm = normalize(X, norm='l2', axis=1)
    Y_norm = normalize(Y, norm='l2', axis=1)
    
    # Uniform weights
    a = np.ones((X_norm.shape[0],)) / X_norm.shape[0]
    b = np.ones((Y_norm.shape[0],)) / Y_norm.shape[0]
    
    # Cost matrix (squared Euclidean distance)
    M = ot.dist(X_norm, Y_norm, metric='euclidean')**2
    
    # Wasserstein-2 distance
    w2 = np.sqrt(ot.emd2(a, b, M))
    
    results.append({'pair': f'{cohort1}_{cohort2}', 'distance': w2})

In [118]:
df_results = pd.DataFrame(results)
distance_matrix = pd.DataFrame(0, index=cohorts, columns=cohorts, dtype=float)

In [ ]:
for row in results:
    c1, c2 = row['pair'].split('_')
    distance_matrix.loc[c1, c2] = row['distance']
    distance_matrix.loc[c2, c1] = row['distance']  # symmetric

mask = np.triu(np.ones_like(distance_matrix, dtype=bool))
plt.figure(figsize=(12, 10))
sns.heatmap(distance_matrix, annot=True, fmt=".2f", cmap='viridis',
            mask=mask, cbar_kws={'label': 'Wasserstein-2 Distance'})
plt.title('Wasserstein-2 Distance Between Cohorts (Lower Triangle Only)')
plt.show()


In [ ]:
# Suppose 'distance_matrix' is already built with labels like 'ClientA_CN', 'ClientB_AD'

# Get unique (cohort, label) groups
groups = df_feats.groupby(['cohort', 'label'])

group_keys = list(groups.groups.keys())  # list of tuples (cohort, label)

results = []

# Compute Wasserstein-2 distance between all pairs of groups
for (c1, l1), (c2, l2) in itertools.combinations(group_keys, 2):
    X = groups.get_group((c1, l1))[features_cols].to_numpy()
    Y = groups.get_group((c2, l2))[features_cols].to_numpy()
    
    # L2-normalize per sample
    X_norm = normalize(X, norm='l2', axis=1)
    Y_norm = normalize(Y, norm='l2', axis=1)
    
    # Uniform weights
    a = np.ones((X_norm.shape[0],)) / X_norm.shape[0]
    b = np.ones((Y_norm.shape[0],)) / Y_norm.shape[0]
    
    # Cost matrix (squared Euclidean)
    M = ot.dist(X_norm, Y_norm, metric='euclidean')**2
    
    # Wasserstein-2 distance
    w2 = np.sqrt(ot.emd2(a, b, M))
    
    results.append({
        'pair': f'{c1}_{l1}-{c2}_{l2}',
        'distance': w2
    })

# Convert to DataFrame and save

# Convert to DataFrame and save
df_results = pd.DataFrame(results)

# Optional: create a heatmap
# Build a square distance matrix
all_keys = [f'{c}_{l}' for c, l in group_keys]
distance_matrix = pd.DataFrame(0, index=all_keys, columns=all_keys, dtype=float)

for row in results:
    g1, g2 = row['pair'].split('-')
    distance_matrix.loc[g1, g2] = row['distance']
    distance_matrix.loc[g2, g1] = row['distance']  # symmetric

plt.figure(figsize=(12, 10))
sns.heatmap(distance_matrix, annot=True, fmt=".2f", cmap='viridis')
plt.title('Wasserstein-2 Distance Between Cohort-Label Groups')
plt.show()

In [ ]:
# Map label numbers to class names
label_map = {0: 'CN', 1: 'AD'}

# Build group keys with class names
all_keys = [f'{c}_{label_map[l]}' for c, l in group_keys]

# Order keys: all CN first, then all AD
all_keys_sorted = sorted(all_keys, key=lambda x: x.split('_')[-1])

# Build a square distance matrix
distance_matrix = pd.DataFrame(0, index=all_keys_sorted, columns=all_keys_sorted, dtype=float)

for row in results:
    g1, g2 = row['pair'].split('-')
    # Replace numeric labels with class names
    g1_name = f'{g1.split("_")[0]}_{label_map[int(g1.split("_")[1])]}'
    g2_name = f'{g2.split("_")[0]}_{label_map[int(g2.split("_")[1])]}'
    
    distance_matrix.loc[g1_name, g2_name] = row['distance']
    distance_matrix.loc[g2_name, g1_name] = row['distance']  # symmetric

# Create a mask for the upper triangle
mask = np.triu(np.ones_like(distance_matrix, dtype=bool))

plt.figure(figsize=(12, 10))
sns.heatmap(distance_matrix, annot=True, fmt=".2f", cmap='viridis',
            mask=mask, cbar_kws={'label': 'Wasserstein-2 Distance'})
plt.title('Wasserstein-2 Distance Between Client-Label Groups (Lower Triangle Only)')
plt.show()

In [ ]:
# Suppose 'distance_matrix' is already built with labels like 'ClientA_CN', 'ClientB_AD'
labels = distance_matrix.index.tolist()
labels = [lbl.replace("_", "-") for lbl in labels]  # Replace underscore with dash for better parsing
# Extract class names from group keys
classes = [lbl.split('-')[-1] for lbl in labels]

from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Polygon, Patch

# Sequential single-hue (blue) colormap: light = small distance, dark = large distance
# Sequential single-hue (gray) colormap: light = small distance, dark = large distance
distance_cmap = LinearSegmentedColormap.from_list(
    'seq_gray', ["#babebe", "#909a99", "#717879", "#4b4f51", "#0b0b0b"]
)

# Categorical colors for the group-pair border only (never used for cell fill)
group_border_colors = {
    'CN_CN': '#1b9e77',  # blue
    'AD_AD': '#7570b3',  # orange
    'CN_AD': '#d95f02',  # aqua (mixed comparison)
}
group_labels = {
    'CN_CN': 'CN vs CN',
    'AD_AD': 'AD vs AD',
    'CN_AD': 'CN vs AD',
}

def group_key(cls_i, cls_j):
    if cls_i == cls_j:
        return f'{cls_i}_{cls_j}'
    return 'CN_AD'

# Inset each border by this many cell-widths so that borders of adjacent
# blocks don't sit exactly on top of each other (which hid whichever was
# drawn first) -- each block's outline shrinks toward its own interior,
# leaving a small gap between neighboring borders.
BORDER_MARGIN = 0.04

def staircase_vertices(r0, c0, k, margin=0.0):
    """Boundary of the strictly-lower-triangular cells of a k x k block
    (block-local origin at row r0, col c0), following the cell edges,
    inset inward by `margin`."""
    m = margin
    verts = [(c0 + m, r0 + 1 + m)]
    for j in range(k - 2):
        verts.append((c0 + j + 1 - m, r0 + j + 1 + m))
        verts.append((c0 + j + 1 - m, r0 + j + 2 + m))
    verts.append((c0 + k - 1 - m, r0 + k - 1 + m))
    verts.append((c0 + k - 1 - m, r0 + k - m))
    verts.append((c0 + m, r0 + k - m))
    return verts

# Mask upper triangle
mask = np.triu(np.ones_like(distance_matrix, dtype=bool))

plt.figure(figsize=(12, 10))
ax = sns.heatmap(distance_matrix, annot=True, fmt=".2f", mask=mask,
                  cmap=distance_cmap, square=True,
                  linewidths=.5, linecolor='white',
                  cbar_kws={'label': '2-Wasserstein Distance'},
                  annot_kws={"size": 16})

# Keep annotation text legible against both light and dark cells
dist_min = distance_matrix.values.min()
dist_max = distance_matrix.values.max()
norm_dist = (distance_matrix.values - dist_min) / (dist_max - dist_min + 1e-8)
for text in ax.texts:
    x, y = text.get_position()
    col, row = int(x), int(y)
    text.set_color('white' if norm_dist[row, col] > 0.5 else 'black')

# Outline each contiguous CN/AD block with a color that identifies the
# comparison type; position already groups them, the border just labels it.
# Diagonal blocks are only half-visible (the mask hides their upper triangle),
# so their outline follows the staircase of visible cells rather than a smooth
# diagonal (which would leave gaps between the line and the cells). clip_on is
# disabled so the outer edges aren't half-clipped by the axes boundary.
boundaries = [0] + [i for i in range(1, len(classes)) if classes[i] != classes[i - 1]] + [len(classes)]
for bi in range(len(boundaries) - 1):
    for bj in range(len(boundaries) - 1):
        if bi < bj:
            continue  # entirely in the masked upper triangle
        r0, r1 = boundaries[bi], boundaries[bi + 1]
        c0, c1 = boundaries[bj], boundaries[bj + 1]
        key = group_key(classes[r0], classes[c0])
        edgecolor = group_border_colors[key]
        if bi == bj:
            k = r1 - r0
            if k < 2:
                continue  # single-cell block: entirely masked, nothing to outline
            verts = staircase_vertices(r0, c0, k, BORDER_MARGIN)
            ax.add_patch(Polygon(verts, closed=True, fill=False, edgecolor=edgecolor,
                                  lw=4, zorder=10, clip_on=False))
        else:
            m = BORDER_MARGIN
            ax.add_patch(plt.Rectangle((c0 + m, r0 + m), (c1 - c0) - 2 * m, (r1 - r0) - 2 * m,
                                        fill=False, edgecolor=edgecolor, lw=4, zorder=10, clip_on=False))

# Legend explaining the border colors
legend_handles = [Patch(facecolor='none', edgecolor=color, linewidth=3, label=group_labels[key])
                  for key, color in group_border_colors.items()]
ax.legend(handles=legend_handles, loc='upper right', frameon=True, framealpha=0.9,
          fontsize=12)

plt.xticks(np.arange(len(labels))+0.5, labels, rotation=90, fontsize=14)
plt.yticks(np.arange(len(labels))+0.5, labels, rotation=0, fontsize=14)
plt.title('Med-SAM2', fontsize=20)
plt.savefig("../../local/output/Med-SAM2_Wasserstein2_distance_heatmap.png", bbox_inches='tight', dpi=300)
plt.show()

# Medical-SAM2

In [123]:
df_feats = pd.read_csv("../../local/output/combined_data_medicalsam2.csv")
# Assume df_feats has 'client' column and 'Feature1'..'Feature384'
features_cols = [f'Feature{i}' for i in range(1, 257)]

# Get unique cohorts
cohorts = df_feats['cohort'].unique()

# Prepare results list
results = []

In [ ]:
# Compute OT distance for each pair
for cohort1, cohort2 in itertools.combinations(cohorts, 2):
    print(f'Computing distance between {cohort1} and {cohort2}')
    X = df_feats[df_feats['cohort'] == cohort1][features_cols].to_numpy()
    Y = df_feats[df_feats['cohort'] == cohort2][features_cols].to_numpy()
    
    # L2-normalize per sample
    X_norm = normalize(X, norm='l2', axis=1)
    Y_norm = normalize(Y, norm='l2', axis=1)
    
    # Uniform weights
    a = np.ones((X_norm.shape[0],)) / X_norm.shape[0]
    b = np.ones((Y_norm.shape[0],)) / Y_norm.shape[0]
    
    # Cost matrix (squared Euclidean distance)
    M = ot.dist(X_norm, Y_norm, metric='euclidean')**2
    
    # Wasserstein-2 distance
    w2 = np.sqrt(ot.emd2(a, b, M))
    
    results.append({'pair': f'{cohort1}_{cohort2}', 'distance': w2})

In [125]:
df_results = pd.DataFrame(results)
distance_matrix = pd.DataFrame(0, index=cohorts, columns=cohorts, dtype=float)

In [ ]:
for row in results:
    c1, c2 = row['pair'].split('_')
    distance_matrix.loc[c1, c2] = row['distance']
    distance_matrix.loc[c2, c1] = row['distance']  # symmetric

mask = np.triu(np.ones_like(distance_matrix, dtype=bool))
plt.figure(figsize=(12, 10))
sns.heatmap(distance_matrix, annot=True, fmt=".2f", cmap='viridis',
            mask=mask, cbar_kws={'label': 'Wasserstein-2 Distance'})
plt.title('Wasserstein-2 Distance Between Cohorts (Lower Triangle Only)')
plt.show()


In [ ]:
# Suppose 'distance_matrix' is already built with labels like 'ClientA_CN', 'ClientB_AD'

# Get unique (cohort, label) groups
groups = df_feats.groupby(['cohort', 'label'])

group_keys = list(groups.groups.keys())  # list of tuples (cohort, label)

results = []

# Compute Wasserstein-2 distance between all pairs of groups
for (c1, l1), (c2, l2) in itertools.combinations(group_keys, 2):
    X = groups.get_group((c1, l1))[features_cols].to_numpy()
    Y = groups.get_group((c2, l2))[features_cols].to_numpy()
    
    # L2-normalize per sample
    X_norm = normalize(X, norm='l2', axis=1)
    Y_norm = normalize(Y, norm='l2', axis=1)
    
    # Uniform weights
    a = np.ones((X_norm.shape[0],)) / X_norm.shape[0]
    b = np.ones((Y_norm.shape[0],)) / Y_norm.shape[0]
    
    # Cost matrix (squared Euclidean)
    M = ot.dist(X_norm, Y_norm, metric='euclidean')**2
    
    # Wasserstein-2 distance
    w2 = np.sqrt(ot.emd2(a, b, M))
    
    results.append({
        'pair': f'{c1}_{l1}-{c2}_{l2}',
        'distance': w2
    })

# Convert to DataFrame and save

# Convert to DataFrame and save
df_results = pd.DataFrame(results)

# Optional: create a heatmap
# Build a square distance matrix
all_keys = [f'{c}_{l}' for c, l in group_keys]
distance_matrix = pd.DataFrame(0, index=all_keys, columns=all_keys, dtype=float)

for row in results:
    g1, g2 = row['pair'].split('-')
    distance_matrix.loc[g1, g2] = row['distance']
    distance_matrix.loc[g2, g1] = row['distance']  # symmetric

plt.figure(figsize=(12, 10))
sns.heatmap(distance_matrix, annot=True, fmt=".2f", cmap='viridis')
plt.title('Wasserstein-2 Distance Between Cohort-Label Groups')
plt.show()

In [ ]:
# Map label numbers to class names
label_map = {0: 'CN', 1: 'AD'}

# Build group keys with class names
all_keys = [f'{c}_{label_map[l]}' for c, l in group_keys]

# Order keys: all CN first, then all AD
all_keys_sorted = sorted(all_keys, key=lambda x: x.split('_')[-1])

# Build a square distance matrix
distance_matrix = pd.DataFrame(0, index=all_keys_sorted, columns=all_keys_sorted, dtype=float)

for row in results:
    g1, g2 = row['pair'].split('-')
    # Replace numeric labels with class names
    g1_name = f'{g1.split("_")[0]}_{label_map[int(g1.split("_")[1])]}'
    g2_name = f'{g2.split("_")[0]}_{label_map[int(g2.split("_")[1])]}'
    
    distance_matrix.loc[g1_name, g2_name] = row['distance']
    distance_matrix.loc[g2_name, g1_name] = row['distance']  # symmetric

# Create a mask for the upper triangle
mask = np.triu(np.ones_like(distance_matrix, dtype=bool))

plt.figure(figsize=(12, 10))
sns.heatmap(distance_matrix, annot=True, fmt=".2f", cmap='viridis',
            mask=mask, cbar_kws={'label': 'Wasserstein-2 Distance'})
plt.title('Wasserstein-2 Distance Between Client-Label Groups (Lower Triangle Only)')
plt.show()

In [ ]:
# Suppose 'distance_matrix' is already built with labels like 'ClientA_CN', 'ClientB_AD'
labels = distance_matrix.index.tolist()
labels = [lbl.replace("_", "-") for lbl in labels]  # Replace underscore with dash for better parsing
# Extract class names from group keys
classes = [lbl.split('-')[-1] for lbl in labels]

from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Polygon, Patch

# Sequential single-hue (blue) colormap: light = small distance, dark = large distance
# Sequential single-hue (gray) colormap: light = small distance, dark = large distance
distance_cmap = LinearSegmentedColormap.from_list(
    'seq_gray', ["#babebe", "#909a99", "#717879", "#4b4f51", "#0b0b0b"]
)

# Categorical colors for the group-pair border only (never used for cell fill)
group_border_colors = {
    'CN_CN': '#1b9e77',  # blue
    'AD_AD': '#7570b3',  # orange
    'CN_AD': '#d95f02',  # aqua (mixed comparison)
}
group_labels = {
    'CN_CN': 'CN vs CN',
    'AD_AD': 'AD vs AD',
    'CN_AD': 'CN vs AD',
}

def group_key(cls_i, cls_j):
    if cls_i == cls_j:
        return f'{cls_i}_{cls_j}'
    return 'CN_AD'

# Inset each border by this many cell-widths so that borders of adjacent
# blocks don't sit exactly on top of each other (which hid whichever was
# drawn first) -- each block's outline shrinks toward its own interior,
# leaving a small gap between neighboring borders.
BORDER_MARGIN = 0.04

def staircase_vertices(r0, c0, k, margin=0.0):
    """Boundary of the strictly-lower-triangular cells of a k x k block
    (block-local origin at row r0, col c0), following the cell edges,
    inset inward by `margin`."""
    m = margin
    verts = [(c0 + m, r0 + 1 + m)]
    for j in range(k - 2):
        verts.append((c0 + j + 1 - m, r0 + j + 1 + m))
        verts.append((c0 + j + 1 - m, r0 + j + 2 + m))
    verts.append((c0 + k - 1 - m, r0 + k - 1 + m))
    verts.append((c0 + k - 1 - m, r0 + k - m))
    verts.append((c0 + m, r0 + k - m))
    return verts

# Mask upper triangle
mask = np.triu(np.ones_like(distance_matrix, dtype=bool))

plt.figure(figsize=(12, 10))
ax = sns.heatmap(distance_matrix, annot=True, fmt=".2f", mask=mask,
                  cmap=distance_cmap, square=True,
                  linewidths=.5, linecolor='white',
                  cbar_kws={'label': '2-Wasserstein Distance'},
                  annot_kws={"size": 16})

# Keep annotation text legible against both light and dark cells
dist_min = distance_matrix.values.min()
dist_max = distance_matrix.values.max()
norm_dist = (distance_matrix.values - dist_min) / (dist_max - dist_min + 1e-8)
for text in ax.texts:
    x, y = text.get_position()
    col, row = int(x), int(y)
    text.set_color('white' if norm_dist[row, col] > 0.5 else 'black')

# Outline each contiguous CN/AD block with a color that identifies the
# comparison type; position already groups them, the border just labels it.
# Diagonal blocks are only half-visible (the mask hides their upper triangle),
# so their outline follows the staircase of visible cells rather than a smooth
# diagonal (which would leave gaps between the line and the cells). clip_on is
# disabled so the outer edges aren't half-clipped by the axes boundary.
boundaries = [0] + [i for i in range(1, len(classes)) if classes[i] != classes[i - 1]] + [len(classes)]
for bi in range(len(boundaries) - 1):
    for bj in range(len(boundaries) - 1):
        if bi < bj:
            continue  # entirely in the masked upper triangle
        r0, r1 = boundaries[bi], boundaries[bi + 1]
        c0, c1 = boundaries[bj], boundaries[bj + 1]
        key = group_key(classes[r0], classes[c0])
        edgecolor = group_border_colors[key]
        if bi == bj:
            k = r1 - r0
            if k < 2:
                continue  # single-cell block: entirely masked, nothing to outline
            verts = staircase_vertices(r0, c0, k, BORDER_MARGIN)
            ax.add_patch(Polygon(verts, closed=True, fill=False, edgecolor=edgecolor,
                                  lw=4, zorder=10, clip_on=False))
        else:
            m = BORDER_MARGIN
            ax.add_patch(plt.Rectangle((c0 + m, r0 + m), (c1 - c0) - 2 * m, (r1 - r0) - 2 * m,
                                        fill=False, edgecolor=edgecolor, lw=4, zorder=10, clip_on=False))

# Legend explaining the border colors
legend_handles = [Patch(facecolor='none', edgecolor=color, linewidth=3, label=group_labels[key])
                  for key, color in group_border_colors.items()]
ax.legend(handles=legend_handles, loc='upper right', frameon=True, framealpha=0.9,
          fontsize=12)

plt.xticks(np.arange(len(labels))+0.5, labels, rotation=90, fontsize=14)
plt.yticks(np.arange(len(labels))+0.5, labels, rotation=0, fontsize=14)
plt.title('Medical-SAM2', fontsize=20)
plt.savefig("../../local/output/Medical-SAM2_Wasserstein2_distance_heatmap.png", bbox_inches='tight', dpi=300)
plt.show()